# Notebook 3: Modeling: KMeans Clustering, Evaluation & Recommendation
---
**Goal:** Train a KMeans clustering model on the cleaned training data, determine the optimal number of clusters, assign cluster labels to both the training and recommendation datasets, and generate song recommendations.

### Steps in this notebook:
1. Load cleaned training data and define feature matrix
2. Find optimal K using Elbow Method (KneeLocator) and Silhouette Score (A MUST TO CHECK WORK)
3. Train final KMeans model
4. Analyze and describe clusters in human terms
5. Apply model to `recommend.csv` and generate recommendations
6. Validate model using `train.csv` confirm clusters are meaningful
7. Save all outputs

---
## Setup: Imports and Data Loading

Import all required libraries including `sklearn` for KMeans and silhouette scoring, `kneed` for automated elbow detection, and `pickle` to load the saved scaler from Notebook 2. We load the cleaned training data and confirm the shape matches the output from the cleaning step.

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
from kneed import KneeLocator

%matplotlib inline
sns.set_style('whitegrid')

# File paths
BASE_PATH = '/Users/saadult/Music Rec Algo/Music-Recommendation-Algorithm/data/'
TRAIN_CLEANED_PATH = BASE_PATH + 'train_cleaned.csv'
TRAIN_RAW_PATH     = BASE_PATH + 'train.csv'
RECOMMEND_PATH     = BASE_PATH + 'recommend.csv'
SCALER_PATH        = BASE_PATH + 'scaler.pkl'

# Load cleaned training data
df = pd.read_csv(TRAIN_CLEANED_PATH)
print(f'Cleaned training data shape: {df.shape}')
df.head()

ModuleNotFoundError: No module named 'kneed'

---
## Step 1: Define Feature Matrix

We separate the label columns (`genre`, `topic`, etc.) from the scaled score columns. Only the scaled score columns are fed into KMeans — the label columns are retained alongside the data so we can interpret and describe the clusters after training.

In [ ]:
# Separate label columns from score columns
# Label columns are used for interpretation only — NOT fed into KMeans
label_cols = ['release_date', 'genre', 'topic', 'age', 'len']
label_cols = [c for c in label_cols if c in df.columns]

score_cols = [c for c in df.columns if c not in label_cols]

# X is the feature matrix KMeans will train on
X = df[score_cols].values

print(f'Label columns (for interpretation): {label_cols}')
print(f'Score columns (fed into KMeans):    {score_cols}')
print(f'Feature matrix X shape: {X.shape} — (songs × features)')

---
## Step 2: Find the Optimal Number of Clusters (K)

The most important decision in KMeans is choosing K — the number of clusters. We use two complementary methods:

**Elbow Method:** Runs KMeans for K=2 through K=12 and plots inertia (total within-cluster distance) vs K. We look for the "elbow" — the bend where adding more clusters stops providing significant improvement. KneeLocator automates this detection mathematically instead of relying on visual judgment.

**Silhouette Score:** Measures how well each song fits its assigned cluster compared to neighboring clusters. Ranges from -1 to 1 — higher is better. We pick the K that produces the highest average silhouette score.

If both methods agree → strong choice. If they disagree → we explain our reasoning.

In [ ]:
# Test K = 2 through 12
# Collects inertia and silhouette score for each value of K
# Note: this cell may take 1-2 minutes to run on 28k songs
K_RANGE = range(2, 13)
inertias = []
silhouette_scores = []

print('Testing K values...')
for k in K_RANGE:
    km = KMeans(
        n_clusters=k,
        init='k-means++',   # smarter initialization than random
        n_init=10,          # run 10 times, keep best result
        random_state=42,    # reproducibility
        max_iter=300
    )
    km.fit(X)
    inertias.append(km.inertia_)

    sil = silhouette_score(
        X, km.labels_,
        sample_size=5000,   # sample for speed on large datasets
        random_state=42
    )
    silhouette_scores.append(sil)
    print(f'  K={k:2d} | inertia={km.inertia_:,.1f} | silhouette={sil:.4f}')

print('Done!')

### 2a. Elbow Method with KneeLocator

The elbow plot shows inertia decreasing as K increases. We use KneeLocator to mathematically identify the sharpest bend in the curve — this is the point of diminishing returns where adding more clusters no longer significantly improves the model's tightness.

In [ ]:
# KneeLocator automatically detects the elbow bend
# curve='convex' and direction='decreasing' match the shape of an inertia curve
kneedle = KneeLocator(
    x=list(K_RANGE),
    y=inertias,
    curve='convex',
    direction='decreasing'
)
elbow_k = kneedle.knee

# Plot the elbow with the detected knee marked
kneedle.plot_knee()
plt.title('Elbow Method — Knee Detected by KneeLocator', fontweight='bold')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.tight_layout()
plt.show()

print(f'KneeLocator detected elbow at K = {elbow_k}')

### 2b. Silhouette Score Plot

The silhouette score measures actual cluster quality — how well-separated and internally consistent the clusters are. We look for the peak value. Note: music data typically produces silhouette scores between 0.1 and 0.3 because songs naturally overlap across lyrical themes. A lower score here does not indicate a failed model — it reflects the nature of the data.

In [ ]:
# Silhouette score plot — look for the peak
# The K with the highest silhouette score has the best cluster separation
best_sil_k = list(K_RANGE)[silhouette_scores.index(max(silhouette_scores))]

plt.figure(figsize=(10, 5))
plt.plot(
    list(K_RANGE),
    silhouette_scores,
    'rs-', markersize=7, linewidth=2
)
plt.axvline(
    x=best_sil_k,
    color='green',
    linestyle='--',
    label=f'Best K = {best_sil_k} (score = {max(silhouette_scores):.4f})'
)
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score — Find the Peak', fontweight='bold')
plt.xticks(list(K_RANGE))
plt.legend()
plt.tight_layout()
plt.show()

print(f'Best K by silhouette score: {best_sil_k}')
print(f'Best silhouette score:      {max(silhouette_scores):.4f}')

### 2c. K Decision

We compare both methods and select the final K. If they agree, that K is used directly. If they disagree, we select the silhouette-recommended K as it measures actual cluster quality rather than just compactness, and note the reasoning below.

In [ ]:
# Compare both methods and finalize K
print(f'KneeLocator (Elbow) suggests: K = {elbow_k}')
print(f'Silhouette Score suggests:    K = {best_sil_k}')
print()

if elbow_k == best_sil_k:
    OPTIMAL_K = elbow_k
    print(f'Both methods agree — strong choice: K = {OPTIMAL_K}')
else:
    # Default to silhouette when methods disagree
    OPTIMAL_K = best_sil_k
    print(f'Methods disagree — using silhouette recommendation: K = {OPTIMAL_K}')
    print('Silhouette is preferred because it measures actual cluster quality,')
    print('not just inertia compactness.')

print(f'\nFINAL CHOICE: K = {OPTIMAL_K}')

**Finding:** *(Update this cell after running the above code with your actual results.)*

Write what K each method recommended, whether they agreed, and which K you chose. This is your answer to **Report Question iii**.

---
## Step 3: Train the Final KMeans Model

We train one final model using the chosen K. Setting `n_init=20` means the algorithm tries 20 different random starting configurations and keeps the best one — this makes the result more stable and reliable than the default of 10.

In [ ]:
# Train final KMeans model with optimal K
# n_init=20 runs 20 initializations and keeps the best — more reliable than default
final_model = KMeans(
    n_clusters=OPTIMAL_K,
    init='k-means++',
    n_init=20,
    random_state=42,
    max_iter=300
)

final_model.fit(X)

# Add cluster labels to the dataframe
df['cluster'] = final_model.labels_

print(f'Model trained with K = {OPTIMAL_K}')
print(f'Final inertia: {final_model.inertia_:,.2f}')
print(f'\nSongs per cluster:')
print(df['cluster'].value_counts().sort_index())

---
## Step 4: Analyze and Describe the Clusters

Now we interpret what each cluster actually means. We look at three things:
1. The average score for each feature per cluster (cluster profile heatmap)
2. What genres ended up in each cluster
3. Sample song titles from each cluster

The goal is to describe each cluster in human terms — for example 'high-obscene, low-sadness songs' or 'older world/life songs'. This is your answer to **Report Question iv**.

In [ ]:
# Cluster profile — average score per feature per cluster
# High values = that cluster's songs strongly express that lyrical theme
cluster_profile = df.groupby('cluster')[score_cols].mean().round(3)
print('Average score per cluster (pre-scaled values shown for readability):')
cluster_profile

### 4a. Cluster Profile Heatmap

Each row is one cluster. Each column is one feature. Dark red cells indicate that cluster scores highly on that theme. This is the primary tool for naming clusters in human terms — scan each row for its darkest cells and use those to describe the cluster.

In [ ]:
# Cluster profile heatmap
# Dark red = cluster scores high on that feature
# Use this to name each cluster in human terms
plt.figure(figsize=(16, max(4, OPTIMAL_K)))
sns.heatmap(
    cluster_profile,
    annot=True,
    fmt='.2f',
    cmap='YlOrRd',
    linewidths=0.5,
    yticklabels=[f'Cluster {i}' for i in range(OPTIMAL_K)]
)
plt.title(
    'Cluster Profiles — Average Score per Feature per Cluster',
    fontweight='bold',
    fontsize=13
)
plt.tight_layout()
plt.show()

**Finding:** *(Update this cell after running the above.)*

Look at each row and identify the 1–2 features with the darkest color. Write a one-sentence description of each cluster. For example:
- **Cluster 0:** High obscene, low sadness — likely hip hop songs
- **Cluster 1:** High sadness, high feelings — emotional ballads
- *(continue for each cluster)*

This is your answer to **Report Question iv**.

### 4b. Genre Breakdown by Cluster

This stacked bar chart shows what proportion of each cluster comes from each genre. If one cluster is almost entirely hip hop, that confirms the model captured genre differences without being told what genre any song belongs to. If clusters are mixed, the model found patterns that cut across genre lines.

In [ ]:
# Genre proportions within each cluster
# normalize='index' shows each row as percentages summing to 100%
if 'genre' in df.columns:
    genre_cluster = pd.crosstab(
        df['cluster'],
        df['genre'],
        normalize='index'
    ).round(2)

    genre_cluster.plot(
        kind='bar',
        stacked=True,
        figsize=(14, 6),
        colormap='tab10'
    )
    plt.xlabel('Cluster')
    plt.ylabel('Proportion of Songs')
    plt.title(
        'Genre Composition Within Each Cluster',
        fontweight='bold'
    )
    plt.legend(
        title='Genre',
        bbox_to_anchor=(1.05, 1),
        loc='upper left'
    )
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

### 4c. Sample Songs from Each Cluster

We print 5 sample songs from each cluster so we can listen to them and verify the cluster descriptions make intuitive sense. If a cluster labeled 'high violence' contains recognizably aggressive songs, the model is working correctly.

In [ ]:
# Load raw train.csv to get artist and track names for display
# (These were dropped before modeling but are useful for interpretation)
df_raw = pd.read_csv(TRAIN_RAW_PATH)
df_raw['cluster'] = df['cluster'].values

display_cols = [
    c for c in ['artist_name', 'track_name', 'genre', 'topic', 'cluster']
    if c in df_raw.columns
]

# Print 5 sample songs per cluster
for cluster_id in sorted(df_raw['cluster'].unique()):
    print(f'\n--- CLUSTER {cluster_id} ---')
    sample = (
        df_raw[df_raw['cluster'] == cluster_id][display_cols]
        .sample(min(5, len(df_raw[df_raw['cluster'] == cluster_id])), random_state=42)
    )
    print(sample.to_string(index=False))

---
## Step 5: Save Trained Model and Labeled Training Data

We save the trained KMeans model using `pickle` so it can be loaded and applied to the recommendation dataset without re-training. We also save the training data with cluster labels appended as a new column.

In [ ]:
# Save the trained KMeans model
MODEL_PATH = BASE_PATH + 'kmeans_model.pkl'
with open(MODEL_PATH, 'wb') as f:
    pickle.dump(final_model, f)
print(f'Model saved to: {MODEL_PATH}')

# Save training data with cluster labels
TRAIN_CLUSTERS_PATH = BASE_PATH + 'train_with_clusters.csv'
df.to_csv(TRAIN_CLUSTERS_PATH, index=False)
print(f'Training data with clusters saved to: {TRAIN_CLUSTERS_PATH}')
print(f'Shape: {df.shape}')

---
## Step 6: Apply Model to Recommend.csv

We load the 11 recommendation test songs, apply the same cleaning and scaling used on the training data, then use the trained model to predict a cluster label for each song. Songs that land in the same cluster share similar lyrical DNA and can be recommended to each other.

**Critical rule:** We use the scaler saved from Notebook 2 — we do NOT re-fit a new scaler on the recommendation data. Re-fitting would put the test data on a different scale than the training data, making the predictions meaningless.

In [ ]:
# Load the 11 recommendation test songs
df_rec = pd.read_csv(RECOMMEND_PATH)
print(f'Recommend dataset shape: {df_rec.shape}')
print(f'Columns: {df_rec.columns.tolist()}')
df_rec.head(20)

In [ ]:
# Load the saved scaler from Notebook 2
# This ensures the recommend data is scaled identically to the training data
with open(SCALER_PATH, 'rb') as f:
    scaler = pickle.load(f)
print('Scaler loaded successfully.')

# Ensure recommend.csv has the same score columns as training data
# Fill any missing columns with 0 (neutral value)
for col in score_cols:
    if col not in df_rec.columns:
        df_rec[col] = 0.0
        print(f'  Added missing column with default 0: {col}')

# Scale the recommend data using the TRAINING scaler (do not re-fit)
X_rec = scaler.transform(df_rec[score_cols])
print(f'\nRecommend feature matrix shape: {X_rec.shape}')

### 6a. Predict Clusters for Recommend Songs

`.predict()` assigns each recommendation song to its nearest cluster centroid. The model already knows where the centroids are from training — it simply measures which centroid each new song is closest to and assigns that cluster label.

In [ ]:
# Predict cluster for each of the 11 recommendation songs
df_rec['cluster'] = final_model.predict(X_rec)

print('Cluster assignments for recommendation songs:')
display_rec_cols = [
    c for c in ['artist_name', 'track_name', 'genre', 'topic', 'cluster']
    if c in df_rec.columns
]
print(df_rec[display_rec_cols].to_string(index=False))

### 6b. Songs Grouped by Cluster

Songs that landed in the same cluster share the most similar lyrical profiles. This grouping is the basis of our recommendations — songs in the same cluster can be recommended to users who enjoy another song in that cluster.

In [ ]:
# Group recommendation songs by their assigned cluster
# Songs in the same cluster are most similar to each other
print('=== RECOMMENDED SONGS GROUPED BY CLUSTER ===')
print('Songs in the same cluster share the most similar lyrical themes.\n')

for cluster_id in sorted(df_rec['cluster'].unique()):
    songs = df_rec[df_rec['cluster'] == cluster_id]
    print(f'--- Cluster {cluster_id} ({len(songs)} song(s)) ---')
    for _, row in songs.iterrows():
        artist = row.get('artist_name', 'Unknown')
        track  = row.get('track_name', 'Unknown')
        genre  = row.get('genre', 'Unknown')
        print(f'  {artist} — {track} ({genre})')
    print()

### 6c. Find Similar Songs from Training Data

For each recommendation song, we find 5 songs from the same cluster in the training data. These are songs with similar lyrical DNA — natural recommendations based on the cluster the user's song belongs to. This directly answers **Report Question v**.

In [ ]:
# Generate song recommendations from the training data
# For each recommend song: find 5 similar songs from the same cluster in train.csv
print('=== SONG RECOMMENDATIONS BASED ON CLUSTER SIMILARITY ===\n')

rec_display_cols = [
    c for c in ['artist_name', 'track_name', 'genre', 'release_date']
    if c in df_raw.columns
]

for _, rec_song in df_rec.iterrows():
    cluster_id  = rec_song['cluster']
    song_name   = rec_song.get('track_name', 'Unknown')
    artist_name = rec_song.get('artist_name', 'Unknown')

    print(f'Because you like: "{song_name}" by {artist_name} → Cluster {cluster_id}')
    print('You might also enjoy:')

    cluster_songs = df_raw[df_raw['cluster'] == cluster_id]
    similar = cluster_songs[rec_display_cols].sample(
        min(5, len(cluster_songs)),
        random_state=42
    )

    for _, s in similar.iterrows():
        print(
            f"  - {s.get('artist_name','?')} — "
            f"{s.get('track_name','?')} "
            f"({s.get('genre','?')})"
        )
    print()

**Finding:** *(Update this cell after running.)*

Note which cluster each recommendation song landed in. Do the groupings make intuitive sense? Did songs that feel similar to you end up in the same cluster? Did any song land somewhere unexpected? This reasoning is your answer to **Report Question v**.

---
## Step 7: Model Validation Using train.csv

We validate the model by checking whether songs from `train.csv` separate into meaningful clusters that align with our EDA expectations. Specifically:
- Do high-obscene training songs mostly fall into one cluster?
- Do cluster topic distributions align with our EDA genre/topic analysis?

This confirms the model is capturing real lyrical patterns rather than making arbitrary groupings.

In [ ]:
# Validate: do topics distribute meaningfully across clusters?
# If songs labeled 'violence' mostly land in the high-violence cluster → model is working
if 'topic' in df_raw.columns:
    topic_cluster = pd.crosstab(
        df_raw['cluster'],
        df_raw['topic'],
        normalize='index'
    ).round(2)

    plt.figure(figsize=(14, max(4, OPTIMAL_K)))
    sns.heatmap(
        topic_cluster,
        annot=True,
        fmt='.2f',
        cmap='YlOrRd',
        linewidths=0.5
    )
    plt.title(
        'Topic Distribution Within Each Cluster — Validation Check',
        fontweight='bold'
    )
    plt.tight_layout()
    plt.show()

In [ ]:
# Validation: does the cluster with highest mean obscene score
# align with what we identified as the hip hop cluster in EDA?
cluster_means = df.groupby('cluster')[score_cols].mean()
top_obscene_cluster = cluster_means['obscene'].idxmax()

print(f'Cluster with highest average obscene score: Cluster {top_obscene_cluster}')
print('\nGenre breakdown of that cluster:')
print(
    df_raw[df_raw['cluster'] == top_obscene_cluster]['genre']
    .value_counts(normalize=True)
    .round(2)
)
print('\n(If hip hop dominates this cluster, the model captured our EDA finding correctly.)')

**Validation finding:** *(Update after running.)*

Note whether the highest-obscene cluster aligns with hip hop dominance (as predicted in EDA). Note whether the topic distribution heatmap shows any clear alignment between lyrical topics and clusters. Strong alignment = the model is working correctly.

---
## Step 8: Save All Outputs

In [ ]:
# Save recommendation data with cluster labels
REC_CLUSTERS_PATH = BASE_PATH + 'recommend_with_clusters.csv'
df_rec.to_csv(REC_CLUSTERS_PATH, index=False)
print(f'Recommend data with clusters saved to: {REC_CLUSTERS_PATH}')
print(f'Shape: {df_rec.shape}')

# Confirm all output files
print('\n=== All output files ===')
print(f'  {MODEL_PATH}')
print(f'  {TRAIN_CLUSTERS_PATH}')
print(f'  {REC_CLUSTERS_PATH}')

---
## Modeling Summary

| Step | Action | Result |
|---|---|---|
| Feature matrix | 11 scaled score columns from Notebook 2 | X shape: ~28k × 11 |
| K selection | Elbow (KneeLocator) + Silhouette Score | K = *[fill in after running]* |
| Model | KMeans (k-means++, n_init=20) | Inertia = *[fill in]* |
| Silhouette score | Average across all songs | *[fill in]* |
| Cluster descriptions | Derived from profile heatmap + sample songs | *[fill in per cluster]* |
| Recommendations | Based on cluster membership in training data | Generated for all 11 songs |

**Report answers covered in this notebook:**
- **Question iii:** Optimal K and how it was determined → Step 2
- **Question iv:** Cluster descriptions in human terms → Step 4
- **Question v:** Song recommendations based on clusters → Step 6